# Ground Truth Cleaning — `ground_truth.csv` → `fixed_ground_truth.csv`

**Purpose:** `ground_truth.csv` is not standard CSV — each row is wrapped in an
extra pair of outer quotes, with the `address` field's own quotes doubled
inside that wrapper. This breaks `pandas.read_csv()` out of the box (it reads
the whole row into the `filename` column). This notebook parses the raw file
correctly, runs a light data-quality audit (logged, not silently fixed), and
writes a clean `fixed_ground_truth.csv` so downstream notebooks (EDA, modelling,
evaluation) can just call `pd.read_csv("fixed_ground_truth.csv")` without
re-handling this every time.

**Scope note:** this only repairs the *parsing* problem (structural/quoting
issue). It does not impute missing addresses or reinterpret ambiguous
birth_dates — those are genuine data issues, not artifacts of a broken parser,
so they're flagged in the audit log for awareness rather than silently
"corrected".

In [1]:
import csv
import re
import pandas as pd
from pathlib import Path

RAW_PATH = Path("ground_truth.csv")          # adjust path if needed
OUTPUT_PATH = Path("fixed_ground_truth.csv")


## Step 1 — Load raw file as plain text

We read the file as raw text rather than through `pd.read_csv`, since the
malformed quoting means the CSV parser can't be trusted to split columns
correctly at this stage.

In [2]:
with open(RAW_PATH, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

# drop the header and any stray blank lines (trailing newline at EOF, etc.)
header = raw_lines[0]
raw_rows = [l for l in raw_lines[1:] if l.strip() != ""]

print(f"Header: {header}")
print(f"Data rows found: {len(raw_rows)}")


Header: filename,name,birth_date,address
Data rows found: 632


## Step 2 — Row parser for the double-quoted format

Observed pattern in the raw file (see EDA / manual inspection):

- Most rows look like: a whole row wrapped in one outer pair of quotes,
  with the address field's own quotes doubled inside that wrapper
  (e.g. `filename,name,birth_date,` followed by an escaped, doubly-quoted
  address containing commas).
- Some rows have **no** outer wrapper at all, e.g. `image_080.jpg,NAME,DATE,ADDR`
  — used whenever no field contains a comma that needs escaping.
- A few rows quote just the `name` field for the same reason (e.g. a name
  containing a comma, like `LAU, TSZ LAN`).

Fix: if a line is wrapped in outer quotes, strip them and un-escape the
doubled inner quotes back to single quotes. What's left is well-formed CSV,
which we then hand to `csv.reader` to split on commas correctly (respecting
any remaining quoted fields, e.g. the LAU case).

In [3]:
def parse_row(line: str) -> list[str]:
    if line.startswith('"') and line.endswith('"'):
        inner = line[1:-1].replace('""', '"')
        fields = next(csv.reader([inner]))
    else:
        fields = next(csv.reader([line]))
    return fields


## Step 3 — Parse every row, with per-row validation

Every row must resolve to exactly 4 fields (`filename, name, birth_date,
address`). Rows that don't are logged as parse failures rather than
silently dropped or force-fit, so nothing goes missing unnoticed.

In [4]:
parsed_records = []
parse_failures = []

for i, line in enumerate(raw_rows, start=1):
    fields = parse_row(line)
    if len(fields) != 4:
        parse_failures.append((i, len(fields), line[:120]))
    else:
        parsed_records.append(fields)

print(f"Successfully parsed: {len(parsed_records)} / {len(raw_rows)}")
print(f"Parse failures: {len(parse_failures)}")

if parse_failures:
    print("\nFailed rows (line_no, n_fields, preview):")
    for fail in parse_failures:
        print(fail)


Successfully parsed: 632 / 632
Parse failures: 0


## Step 4 — Build the clean DataFrame

In [5]:
df = pd.DataFrame(parsed_records, columns=["filename", "name", "birth_date", "address"])
df["filename"] = df["filename"].str.strip()
df["name"] = df["name"].str.strip()
df["birth_date"] = df["birth_date"].str.strip()
df["address"] = df["address"].str.strip()

print(df.shape)
df.head(10)


(632, 4)


,filename,name,birth_date,address
0,image_001.jpg,HAMZAH BIN KAMMAPU,1965-02-11,"PT 1160 P, JALAN KENANGA, 20400 KUALA TERENGGA..."
1,image_002.jpg,RAZALI BIN AHMAD,1959-02-24,"167, KAMPUNG BANGGOL AIR LILEH, BATU ENAM, 212..."
2,image_003.jpg,NOR ATHIRAH NAJWA BINTI RAZALI,2000-12-30,"167, KAMPUNG BANGGOL AIR LILEH, BATU 6, 21200 ..."
3,image_004.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
4,image_005.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
5,image_006.jpg,MUHAMAD ZAMRI BIN SHAFEE,1986-06-13,MARKAS TENTERA DARAT CAWANGAN SUMBER MANUSIA K...
6,image_007.jpg,MOHAMAD FADZALIISAM BIN ITHNIN,1979-07-19,"K-G-16 JALAN 2/6, TAMAN SETAPAK INDAH, 53300 K..."
7,image_008.jpg,MOHAMAD ADAM HARRIS BIN MOHAMAD FADZALIISAM,2016-04-07,"BLOK E5-1-1, TAMAN MELATI, SETAPAK, 53100 KUAL..."
8,image_009.jpg,AFFANDY BIN OTHMAN,1981-09-19,"NO 1592, JALAN SIRAM, 12100 BUTTERWORTH, PULAU..."
9,image_010.jpg,FATIN NUR NAJWA BINTI FAMY,2000-08-12,"NO 29 JALAN PLATINUM 7/50, SEKSYEN 7, 40000 SH..."


## Step 5 — Data quality audit (log only, no silent fixes)

These are genuine data characteristics/issues to be aware of downstream —
not parsing bugs, so they aren't altered here. This complements the E2
ground truth audit rather than replacing it.

In [6]:
audit_log = {}

# duplicate filenames
audit_log["duplicate_filenames"] = int(df["filename"].duplicated().sum())

# empty fields
audit_log["empty_name"] = int((df["name"] == "").sum())
audit_log["empty_birth_date"] = int((df["birth_date"] == "").sum())
audit_log["empty_address"] = int((df["address"] == "").sum())

# birth_date not in strict YYYY-MM-DD format (e.g. year-only values like "1983")
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
non_standard_dates = df[~df["birth_date"].str.match(date_pattern)]
audit_log["non_standard_birth_date_rows"] = int(len(non_standard_dates))

print("Audit summary:")
for k, v in audit_log.items():
    print(f"  {k}: {v}")

if len(non_standard_dates) > 0:
    print("\nSample non-standard birth_date rows:")
    print(non_standard_dates[["filename", "name", "birth_date"]].head(10).to_string(index=False))


Audit summary:
  duplicate_filenames: 0
  empty_name: 0
  empty_birth_date: 0
  empty_address: 520
  non_standard_birth_date_rows: 20

Sample non-standard birth_date rows:
     filename             name birth_date
image_273.jpg ABDELLAH SLIMANI       1983
image_274.jpg ABDELLAH SLIMANI       1983
image_275.jpg ABDELLAH SLIMANI       1983
image_276.jpg ABDELLAH SLIMANI       1983
image_277.jpg ABDELLAH SLIMANI       1983
image_278.jpg ABDELLAH SLIMANI       1983
image_279.jpg ABDELLAH SLIMANI       1983
image_280.jpg ABDELLAH SLIMANI       1983
image_281.jpg ABDELLAH SLIMANI       1983
image_282.jpg ABDELLAH SLIMANI       1983


## Step 6 — Save `fixed_ground_truth.csv`

Standard, well-quoted CSV — safe to read anywhere downstream with a plain
`pd.read_csv()`, no special handling required.

In [7]:
df.to_csv(OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")


Saved: /home/claude/nb/fixed_ground_truth.csv


## Step 7 — Sanity check

Reload with plain `pd.read_csv` (no custom parsing) to confirm the output
is now standard, well-formed CSV.

In [8]:
check_df = pd.read_csv(OUTPUT_PATH)
print(check_df.shape)
assert check_df.shape == df.shape, "Row/column count mismatch after reload!"
assert list(check_df.columns) == ["filename", "name", "birth_date", "address"], "Column mismatch!"
print("Reload check passed — fixed_ground_truth.csv is standard-parseable.")
check_df.head(10)


(632, 4)
Reload check passed — fixed_ground_truth.csv is standard-parseable.


,filename,name,birth_date,address
0,image_001.jpg,HAMZAH BIN KAMMAPU,1965-02-11,"PT 1160 P, JALAN KENANGA, 20400 KUALA TERENGGA..."
1,image_002.jpg,RAZALI BIN AHMAD,1959-02-24,"167, KAMPUNG BANGGOL AIR LILEH, BATU ENAM, 212..."
2,image_003.jpg,NOR ATHIRAH NAJWA BINTI RAZALI,2000-12-30,"167, KAMPUNG BANGGOL AIR LILEH, BATU 6, 21200 ..."
3,image_004.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
4,image_005.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
5,image_006.jpg,MUHAMAD ZAMRI BIN SHAFEE,1986-06-13,MARKAS TENTERA DARAT CAWANGAN SUMBER MANUSIA K...
6,image_007.jpg,MOHAMAD FADZALIISAM BIN ITHNIN,1979-07-19,"K-G-16 JALAN 2/6, TAMAN SETAPAK INDAH, 53300 K..."
7,image_008.jpg,MOHAMAD ADAM HARRIS BIN MOHAMAD FADZALIISAM,2016-04-07,"BLOK E5-1-1, TAMAN MELATI, SETAPAK, 53100 KUAL..."
8,image_009.jpg,AFFANDY BIN OTHMAN,1981-09-19,"NO 1592, JALAN SIRAM, 12100 BUTTERWORTH, PULAU..."
9,image_010.jpg,FATIN NUR NAJWA BINTI FAMY,2000-08-12,"NO 29 JALAN PLATINUM 7/50, SEKSYEN 7, 40000 SH..."
